# 🇬🇧 Notebook B3 — Path B: Train English Expert (LoRA)

**Path B Concrete Specs — Stage 3: Monolingual English Expert**

In PolyWhisper's MoLE architecture, two dedicated monolingual LoRA adapters are trained:
1. **Bengali Expert** (**Notebook B2**) — specializes in Bengali acoustics & vocabulary.
2. **English Expert** (this notebook) — specializes in English acoustics & vocabulary.

> ⚡ **Genuinely Parallel**: Stages 2 and 3 have **zero dependencies** on each other. Run this notebook on a separate cloud GPU at the same time Notebook B2 is running!

### Specs:
- **Backbone**: Frozen `SayedShaun/bengali-whisper-medium`
- **Data**: English-dense / English subset (`train_english_expert.csv` from `data_path_b/`)
- **Processor / Decoder Language**: English (`language='English'`, `task='transcribe'`) to specialize in Latin-script acoustic modeling
- **LoRA Config**: `r=32`, `alpha=64`, `target_modules=['q_proj', 'v_proj']`, `dropout=0.05`
- **Output**: `My Drive/hoichoi/adapters_path_b/english_expert/best`

**Prerequisite:** Notebook B1 must have completed and generated `data_path_b/train_english_expert.csv`.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive + Verify Data
# Main Google Drive Folder: https://drive.google.com/drive/folders/1zqBtJvJ8ZwZNNpLCiS2F-x-yBpMxYXfE?usp=sharing
# data_path_b Folder: https://drive.google.com/drive/folders/1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D?usp=sharing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

DRIVE_ROOT      = '/content/drive/MyDrive/hoichoi'
DRIVE_DATA_B    = DRIVE_ROOT + '/data_path_b'
TRAIN_CSV       = DRIVE_DATA_B + '/train_english_expert.csv'
VAL_CSV         = DRIVE_DATA_B + '/val_english_expert.csv'
ADAPTER_DIR     = DRIVE_ROOT + '/adapters_path_b/english_expert'
DATA_B_DIR_ID   = '1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D'

os.makedirs(DRIVE_DATA_B, exist_ok=True)
os.makedirs(ADAPTER_DIR,  exist_ok=True)

# Direct check for English expert data in data_path_b
if not (os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV)):
    print(f'📥 Fetching data_path_b into {DRIVE_DATA_B} from Drive folder link...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    try:
        gdown.download_folder(id=DATA_B_DIR_ID, output=DRIVE_DATA_B, quiet=False, use_cookies=False)
    except Exception as e:
        print('Download notice:', e)

assert os.path.exists(TRAIN_CSV), f'❌ {TRAIN_CSV} not found! Check data_path_b on Drive.'
assert os.path.exists(VAL_CSV),   f'❌ {VAL_CSV} not found! Check data_path_b on Drive.'

print('✅ English Expert data verified:')
print('   Main folder →', DRIVE_ROOT)
print('   Train CSV   →', TRAIN_CSV, f'({os.path.getsize(TRAIN_CSV)} bytes)')
print('   Val CSV     →', VAL_CSV, f'({os.path.getsize(VAL_CSV)} bytes)')
print('   Adapter Out →', ADAPTER_DIR)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Dependencies & GPU Verification
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
print('Installing transformers, peft, accelerate, librosa, soundfile ...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0,<0.14.0', 'accelerate>=0.29.0',
    'librosa', 'soundfile'])
print('✅ Packages installed successfully.')

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    print('⚠️ No GPU found! Please navigate to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Dataset & Audio Collator
# Loads exact audio segment slice via start_sec & end_sec
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, unicodedata
import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset

class EnglishExpertDataset(Dataset):
    def __init__(self, csv_path, feature_extractor, tokenizer, sr=16000):
        self.df  = pd.read_csv(csv_path)
        self.df  = self.df[self.df['audio_path'].notna()].reset_index(drop=True)
        self.fe  = feature_extractor
        self.tok = tokenizer
        self.sr  = sr
        self.max_len = 30 * sr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s_sec = float(row.get('start_sec', 0.0))
        e_sec = float(row.get('end_sec', 0.0))
        dur   = (e_sec - s_sec) if e_sec > s_sec else None
        
        apath = str(row['audio_path'])
        if not os.path.exists(apath):
            fname = os.path.basename(apath)
            for cdir in [
                '/content/drive/MyDrive/hoichoi/mucs_raw/train',
                '/content/drive/MyDrive/hoichoi/mucs_raw',
                '/content/drive/MyDrive/mucs_raw/train',
                '/content/mucs_raw/train',
                '/content'
            ]:
                candidate = os.path.join(cdir, fname)
                if os.path.exists(candidate):
                    apath = candidate
                    break
        
        try:
            audio, _ = librosa.load(apath, sr=self.sr, mono=True,
                                    offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
            if len(audio) == 0:
                audio = np.zeros(self.sr, dtype=np.float32)
        except Exception:
            audio = np.zeros(self.sr, dtype=np.float32)
        
        audio = audio[:self.max_len]
        feats = self.fe(audio, sampling_rate=self.sr, return_tensors='pt').input_features.squeeze(0)
        
        text   = unicodedata.normalize('NFC', str(row['transcript']))
        labels = self.tok(text, return_tensors='pt').input_ids.squeeze(0)
        labels = labels[:448]  # Whisper positional embedding limit
        return {'input_features': feats, 'labels': labels}

def collate_expert(batch):
    feats  = torch.stack([b['input_features'] for b in batch])
    labels = [b['labels'] for b in batch]
    maxlen = max(max(l.size(0) for l in labels), 1)
    padded = torch.full((len(labels), maxlen), -100, dtype=torch.long)
    for i, l in enumerate(labels):
        padded[i, :l.size(0)] = l
    return {'input_features': feats, 'labels': padded}

print('✅ English Expert Dataset and collator defined')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — English Expert Hyperparameters
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = dict(
    base_model   = 'SayedShaun/bengali-whisper-medium',
    train_csv    = TRAIN_CSV,
    val_csv      = VAL_CSV,
    adapter_out  = ADAPTER_DIR,
    batch_size   = 8,       # If OOM, reduce to 4
    grad_accum   = 4,       # Effective batch size = 32
    max_steps    = 500,
    lr           = 1e-4,
    lora_r       = 32,
    lora_alpha   = 64,
    lora_dropout = 0.05,
    plateau_pat  = 5,       # Early stopping patience
)

print('Configuration for English Expert:')
for k, v in CFG.items():
    print(f'  {k:<14} : {v}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Training Loop for English Expert
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch
from torch.utils.data import DataLoader
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training English Expert on: {device}')

# 1. Load Whisper processor specialized for English transcription
processor = WhisperProcessor.from_pretrained(
    CFG['base_model'], language='English', task='transcribe')

# 2. Load Base Model and configure LoRA
base_model = WhisperForConditionalGeneration.from_pretrained(
    CFG['base_model'],
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
if hasattr(base_model, 'enable_input_require_grads'):
    base_model.enable_input_require_grads()

# Bypass Colab torchao/peft probe incompatibility
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
except Exception:
    pass

lora_cfg = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=CFG['lora_dropout'],
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM)

model = get_peft_model(base_model, lora_cfg).to(device)

# Forward fix: bypass duplicate input_ids conflict
model.forward = lambda **kwargs: model.base_model(**{k: v for k, v in kwargs.items() if k != 'input_ids'})
model.print_trainable_parameters()

# 3. DataLoaders
train_ds = EnglishExpertDataset(CFG['train_csv'], processor.feature_extractor, processor.tokenizer)
val_ds   = EnglishExpertDataset(CFG['val_csv'],   processor.feature_extractor, processor.tokenizer)
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  collate_fn=collate_expert, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, collate_fn=collate_expert, num_workers=0)
print(f'Data: {len(train_ds)} train utterances | {len(val_ds)} validation utterances')

# 4. Optimizer and Mixed Precision Scaler
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG['lr'], weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

best_val_loss = float('inf')
patience_cnt  = 0
global_step   = 0
accum_step    = 0
ckpt_interval = max(1, CFG['max_steps'] // 5)

def evaluate_loss():
    model.eval()
    tot_loss, n_items = 0.0, 0
    with torch.no_grad():
        for batch in val_dl:
            f = batch['input_features'].to(device)
            l = batch['labels'].to(device)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                out = model.base_model(input_features=f, labels=l)
            tot_loss += out.loss.item() * f.size(0)
            n_items  += f.size(0)
    model.train()
    return tot_loss / max(n_items, 1)

model.train()
print(f'\n🚀 Training English Expert for {CFG["max_steps"]} max steps (eval every {ckpt_interval}) ...\n')

for epoch in range(9999):
    for batch in train_dl:
        if global_step >= CFG['max_steps']: break
        
        feats  = batch['input_features'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            loss = model.base_model(input_features=feats, labels=labels).loss / CFG['grad_accum']
            
        scaler.scale(loss).backward()
        accum_step += 1
        
        if accum_step % CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            global_step += 1
            accum_step = 0
            
            if global_step % 10 == 0:
                cur_loss = loss.item() * CFG['grad_accum']
                print(f'  step {global_step:>4}/{CFG["max_steps"]} | train_loss: {cur_loss:.4f}')
                
            if global_step % ckpt_interval == 0 or global_step == CFG['max_steps']:
                v_loss = evaluate_loss()
                print(f'  ✦ step {global_step:>4} | val_loss: {v_loss:.4f}')
                
                ckpt_dir = f"{CFG['adapter_out']}/ckpt_step{global_step}"
                model.save_pretrained(ckpt_dir)
                processor.save_pretrained(ckpt_dir)
                
                if v_loss < best_val_loss - 1e-4:
                    best_val_loss = v_loss
                    patience_cnt  = 0
                    best_dir = f"{CFG['adapter_out']}/best"
                    model.save_pretrained(best_dir)
                    processor.save_pretrained(best_dir)
                    print(f'    ✅ New best English Expert saved! (val_loss: {best_val_loss:.4f})')
                else:
                    patience_cnt += 1
                    print(f'    No improvement ({patience_cnt}/{CFG["plateau_pat"]})')
                    if patience_cnt >= CFG['plateau_pat']:
                        print(f'\n⏹ Plateau reached — stopping early at step {global_step}')
                        break
    else:
        continue
    break

print('=' * 65)
print(f'✅ English Expert Training Complete! Best val_loss: {best_val_loss:.4f}')
print(f'   Saved to → {CFG["adapter_out"]}/best')
print('   👉 Ensure Notebook B2 (Bengali Expert) is also finished before running Notebook B4 (Router).')
print('=' * 65)